# Tables

All tables for the paper. Each is displayed here and written to `../tables/` as `.csv` and a booktabs `.tex` fragment.

Most results are read from the CSVs written by `02_training_and_testing_the_ann.ipynb` and `03_ann_vs_linear_models.ipynb` (in `tables/`, next to the tables built from them). The free-slope circular (Möbius) baselines, and the leave-one-wind-speed-out baselines, are not among those results, so they are fit here with the same trajectory-grouped split (seed 42, 33% test) and the same evaluation protocol as the OLS baselines.

1. Linear / circular baselines and the ANN, by regime
2. Per-trajectory error distribution, all models
3. ANN error by regime
4. Leave-one-wind-speed-out (ANN and baselines)
5. Robustness across training seeds
6. Architecture selection (grouped CV)
7. Ensemble-agreement calibration
8. ANN error vs. trajectory covariates
9. Best baselines and the ANN on all held-out frames and on confident frames (Figure 6)

In [1]:
import os, sys
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import spearmanr
from sklearn.model_selection import GroupShuffleSplit
sys.path.insert(0, os.path.abspath('../utils'))
from utils import circular_distance
from IPython.display import display

D = '../trainingdata/04_trainingdata/'
OUT = '../tables/'
os.makedirs(OUT, exist_ok=True)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', None)
REGIMES = ['upwind', 'crosswind', 'downwind']
THRESH = 15.0

def classify_direction(a):
    deg = np.degrees(np.arctan2(np.sin(a), np.cos(a)))
    return np.where(np.abs(deg) <= 45, 'downwind', np.where(np.abs(np.abs(deg) - 180) <= 45, 'upwind', 'crosswind'))

def bootstrap_ci(v, n_boot=5000, seed=0):
    v = np.asarray(v); rng = np.random.default_rng(seed)
    med = np.median(rng.choice(v, size=(n_boot, len(v)), replace=True), axis=1)
    lo, hi = np.percentile(med, [2.5, 97.5])
    return float(np.median(v)), float(lo), float(hi)

def ci_str(m, lo, hi): return f'{m:.1f} [{lo:.1f}, {hi:.1f}]'

FAMILY = {'linear': 'Linear OLS', 'circular': 'Circular OLS', 'mobius': 'Möbius circular regression', 'ANN': 'ANN'}
INPUT_LABEL = {'thrust': 'thrust', 'air': 'air velocity', 'ground': 'ground velocity'}

def inputs_label(key):
    """'circular: thrust+air' -> 'thrust + air velocity'; the ANN's inputs are air and ground velocity"""
    if key == 'ANN': return 'air velocity + ground velocity'
    return ' + '.join(INPUT_LABEL[q] for q in key.split(': ')[1].split('+'))

def model_label(key): return 'ANN' if key == 'ANN' else f"{FAMILY[key.split(':')[0]]} ({inputs_label(key)})"

def esc(s): return str(s).replace('_', r'\_').replace('%', r'\%').replace('&', r'\&')

def save(df, name, caption, label, align=None):
    df.to_csv(f'{OUT}{name}.csv', index=False)
    align = align or 'l' + 'r' * (df.shape[1] - 1)
    lines = [r'\begin{table}[h!]', r'    \centering', f'    \\caption{{{caption}}}', f'    \\label{{tab:{label}}}',
             f'    \\begin{{tabular}}{{@{{}}{align}@{{}}}}', r'        \toprule',
             '        ' + ' & '.join(esc(c) for c in df.columns) + r' \\', r'        \midrule']
    for _, r in df.iterrows():
        lines.append('        ' + ' & '.join(esc(x) for x in r.values) + r' \\')
    lines += [r'        \bottomrule', r'    \end{tabular}', r'\end{table}']
    open(f'{OUT}{name}.tex', 'w').write('\n'.join(lines) + '\n')
    display(df)

/Users/florisvb/src/drosophila_body_orientation_predictor/.venv/lib/python3.12/site-packages/pynumdiff/__init__.py:9: UserWarning: tvrdiff, robustdiff, and lineardiff not available due to lack of convex solver. To use those, install CVXPY.
  warn("tvrdiff, robustdiff, and lineardiff not available due to lack of convex solver. To use those, install CVXPY.")


## Setup: split, and linear / circular / Möbius baselines

Baselines: *linear* OLS regresses the heading unit vector on the Cartesian components of each kinematic vector; *circular* OLS regresses it on the unit direction only (degree-1 trigonometric polynomial). The additional **Möbius** model is the free-slope circular-circular regression from `03_ann_vs_linear_models.ipynb`: $\hat\theta = \mu + 2\arctan(\beta\tan\frac{\theta_x-\nu}{2})$, fit by minimising $\sum(1-\cos e)$. It takes a single input angle, so it is fit once per kinematic vector.

In [2]:
data = pd.read_csv(D + 'all_wind_heading_and_trajectories_augmented_corrected_filtered_headx_heady.csv')
ids = data['trajec_objid'].unique()
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.33, random_state=42).split(np.zeros((len(ids), 1)), groups=ids))
train_df = data[data['trajec_objid'].isin(set(ids[tr]))].reset_index(drop=True)
test_df  = data[data['trajec_objid'].isin(set(ids[te]))].reset_index(drop=True)
print(len(train_df), 'train frames,', len(test_df), 'test frames')

QUANT = {'thrust': ('thrust', 'thrust_angle'), 'air': ('airspeed', 'airspeed_angle'), 'ground': ('groundspeed', 'groundspeed_angle')}

def design(d, combo, kind):
    cols = [np.ones(len(d))]
    for q in combo:
        m, a = QUANT[q]
        if kind == 'linear': cols += [d[m].values * np.cos(d[a].values), d[m].values * np.sin(d[a].values)]
        else:                cols += [np.cos(d[a].values), np.sin(d[a].values)]
    return np.column_stack(cols)

def ols_angle(train, test, combo, kind):
    Xtr, Xte = design(train, combo, kind), design(test, combo, kind)
    coef = np.linalg.lstsq(Xtr, train[['heading_angle_x', 'heading_angle_y']].values, rcond=None)[0]
    p = Xte @ coef
    return np.arctan2(p[:, 1], p[:, 0])

def circ_mean(a): return np.arctan2(np.sin(a).mean(), np.cos(a).mean())
def mobius(x, mu, nu, lb): return mu + 2 * np.arctan(np.exp(lb) * np.tan((x - nu) / 2))

def fit_mobius(x, y):
    best = None
    for nu0 in np.linspace(0, 2 * np.pi, 8, endpoint=False):
        for lb0 in (-1.0, 0.0, 1.0, 2.0):
            r = minimize(lambda p: np.mean(1 - np.cos(y - mobius(x, *p))), [circ_mean(y - x), nu0, lb0],
                         method='Nelder-Mead', options=dict(xatol=1e-6, fatol=1e-10, maxiter=4000))
            if best is None or r.fun < best.fun: best = r
    return best.x

def mobius_angle(train, test, q):
    a = QUANT[q][1]
    return mobius(test[a].values, *fit_mobius(train[a].values, train['heading_angle'].values))

def frame_errors(test, pred_angle):
    err = np.degrees(circular_distance(test['heading_angle'].values, pred_angle))
    return pd.DataFrame({'trajec_objid': test['trajec_objid'].values, 'err': err,
                         'regime': classify_direction(test['groundspeed_angle'].values)})

def summarize(fdf):
    """per-trajectory medians (pooled and per regime), as in the paper"""
    pooled = fdf.groupby('trajec_objid')['err'].median()
    by_reg = fdf.groupby(['regime', 'trajec_objid'])['err'].median().reset_index()
    out = {'overall': pooled.median()}
    for r in REGIMES: out[r] = by_reg.loc[by_reg['regime'] == r, 'err'].median()
    return out, pooled, by_reg

# Möbius baselines on the paper's split
mobius_rows, mobius_traj = [], {}
for q in QUANT:
    s, pooled, _ = summarize(frame_errors(test_df, mobius_angle(train_df, test_df, q)))
    name = f'mobius: {q}'
    mobius_rows.append({'model': name, **s}); mobius_traj[name] = pooled
print(pd.DataFrame(mobius_rows))

# sanity check: reproduce one saved OLS baseline
ref = pd.read_csv(OUT + 'ols_baselines_by_regime.csv').set_index('model').loc['linear: thrust+air', 'overall']
chk, _, _ = summarize(frame_errors(test_df, ols_angle(train_df, test_df, ('thrust', 'air'), 'linear')))
print(f"check linear: thrust+air overall: saved {ref:.3f}, recomputed {chk['overall']:.3f}")

75141 train frames, 36942 test frames


            model    overall    upwind  crosswind   downwind
0  mobius: thrust  15.216739  9.407608  21.491239  32.588764
1     mobius: air   8.321766  5.402999  13.876132  24.267882
2  mobius: ground  15.910523  4.981799  31.280659  62.756946
check linear: thrust+air overall: saved 9.633, recomputed 9.633


## Table 1 — Linear, circular and Möbius baselines vs. the ANN, by flight regime

Median per-trajectory angular error (deg) on the held-out test set. Regimes are classified from the course direction (`groundspeed_angle`, the direction of travel) relative to the wind axis (downwind $|\theta|\le45^\circ$, upwind within $45^\circ$ of $180^\circ$, otherwise crosswind; the wind blows toward +x).

In [3]:
ann_frames = pd.read_csv(OUT + 'ann_by_regime.csv')
ann_s, ann_pooled, ann_by_reg = summarize(ann_frames)
base = pd.read_csv(OUT + 'ols_baselines_by_regime.csv')
t1 = pd.concat([base, pd.DataFrame(mobius_rows), pd.DataFrame([{'model': 'ANN', **ann_s}])], ignore_index=True)
t1['family'] = t1['model'].str.split(':').str[0].map(FAMILY)
t1['inputs'] = t1['model'].map(inputs_label)
t1 = t1.sort_values('overall', ascending=False).reset_index(drop=True)
table1 = t1[['family', 'inputs', 'overall', 'upwind', 'crosswind', 'downwind']].copy()
table1.columns = ['Model', 'Inputs', 'Overall', 'Upwind', 'Crosswind', 'Downwind']
best_idx = {c: t1[t1['family'] != 'ANN'][c.lower()].idxmin() for c in ['Overall', 'Upwind', 'Crosswind', 'Downwind']}
print('best non-ANN baseline per column:', {c: (t1.loc[i, 'model'], round(t1.loc[i, c.lower()], 2)) for c, i in best_idx.items()})
for c in ['Overall', 'Upwind', 'Crosswind', 'Downwind']: table1[c] = table1[c].map('{:.1f}'.format)
save(table1, 'table1_baselines_by_regime',
     'Median per-trajectory angular error (deg) of every linear, circular and M\\"obius baseline and of the ANN on the held-out test trajectories, overall and by flight regime. Circular OLS and M\\"obius regressions use only the direction of each input; linear OLS and the ANN use its magnitude and direction. Sorted by overall error, worst first.',
     'baseline_regime', align='llrrrr')

best non-ANN baseline per column: {'Overall': ('mobius: air', 8.32), 'Upwind': ('linear: air+ground', 3.99), 'Crosswind': ('mobius: air', 13.88), 'Downwind': ('circular: thrust+air+ground', 23.83)}


,Model,Inputs,Overall,Upwind,Crosswind,Downwind
0,Circular OLS,ground velocity,16.8,4.4,27.6,72.9
1,Möbius circular regression,ground velocity,15.9,5.0,31.3,62.8
2,Möbius circular regression,thrust,15.2,9.4,21.5,32.6
3,Linear OLS,thrust,15.0,7.7,23.7,40.5
4,Circular OLS,thrust,13.9,7.2,26.4,32.6
5,Linear OLS,ground velocity,11.0,4.0,20.3,40.9
6,Circular OLS,thrust + ground velocity,10.7,4.7,21.8,29.7
7,Linear OLS,air velocity + ground velocity,10.2,4.0,19.5,37.3
8,Linear OLS,air velocity,10.0,4.4,18.9,38.6
9,Circular OLS,air velocity,9.8,4.5,23.8,25.3


## Table 2 — Per-trajectory error distribution, all models

Median with 95% bootstrap CI (resampling trajectories), mean, interquartile range, 90th percentile and fraction of trajectories with error above 15°.

In [4]:
def dist_row(name, v):
    v = np.asarray(v); m, lo, hi = bootstrap_ci(v)
    return {'_k': m, 'Model': name, 'n': len(v), 'Median [95% CI]': ci_str(m, lo, hi), 'Mean': f'{v.mean():.1f}',
            'IQR': f'{np.percentile(v, 25):.1f}–{np.percentile(v, 75):.1f}', 'Q90': f'{np.percentile(v, 90):.1f}',
            f'% > {THRESH:g}°': f'{100 * (v > THRESH).mean():.1f}'}

per_traj_all = pd.read_csv(OUT + 'trajectory_level_errors_all_models.csv')
rows = [dist_row(model_label(m), g['err']) for m, g in per_traj_all.groupby('model')]
rows += [dist_row(model_label(m), v.values) for m, v in mobius_traj.items()]
table2 = pd.DataFrame(rows)
table2 = table2.sort_values('_k', ascending=False).drop(columns='_k').reset_index(drop=True)
save(table2, 'table2_error_distribution',
     'Distribution of per-trajectory median angular error (deg) on the held-out test set for the ANN and all baselines.',
     'error_distribution', align='lrrrrrr')

,Model,n,Median [95% CI],Mean,IQR,Q90,% > 15°
0,Circular OLS (ground velocity),1096,"16.8 [14.5, 19.5]",32.1,4.3–47.3,91.4,52.1
1,Möbius circular regression (ground velocity),1096,"15.9 [13.5, 18.1]",28.9,5.4–39.3,62.9,51.0
2,Möbius circular regression (thrust),1096,"15.2 [13.6, 16.7]",22.3,7.7–29.8,50.3,50.2
3,Linear OLS (thrust),1096,"15.0 [13.6, 16.6]",24.1,7.0–35.8,57.0,49.8
4,Circular OLS (thrust),1096,"13.9 [12.7, 15.4]",22.5,6.5–34.0,53.3,47.6
5,Linear OLS (ground velocity),1096,"11.0 [9.6, 12.7]",22.7,4.1–31.0,53.6,43.1
6,Circular OLS (thrust + ground velocity),1096,"10.7 [9.8, 12.0]",19.1,4.6–27.5,46.0,41.4
7,Linear OLS (air velocity + ground velocity),1096,"10.2 [9.0, 11.7]",20.9,4.0–28.2,49.8,41.1
8,Linear OLS (air velocity),1096,"10.0 [8.9, 11.4]",20.7,4.4–29.4,50.3,41.1
9,Circular OLS (air velocity),1096,"9.8 [8.6, 10.8]",19.8,4.4–25.9,49.8,40.3


## Table 3 — ANN error by flight regime

Per-trajectory median error within each regime (a trajectory contributes to every regime it visits), and the best baseline in that regime for reference.

In [5]:
def regime_row(label, v):
    v = np.asarray(v); m, lo, hi = bootstrap_ci(v)
    return {'Regime': label, 'n trajectories': len(v), 'Median [95% CI]': ci_str(m, lo, hi), 'Mean': f'{v.mean():.1f}',
            'IQR': f'{np.percentile(v, 25):.1f}–{np.percentile(v, 75):.1f}', f'% > {THRESH:g}°': f'{100 * (v > THRESH).mean():.1f}'}

rows = [regime_row('Overall', ann_pooled.values)]
rows += [regime_row(r.capitalize(), ann_by_reg.loc[ann_by_reg['regime'] == r, 'err']) for r in REGIMES]
table3 = pd.DataFrame(rows)
nb_base = t1[t1['family'] != 'ANN']
table3['Best baseline (median)'] = [f"{nb_base[c].min():.1f} ({FAMILY[(k := nb_base.loc[nb_base[c].idxmin(), 'model']).split(':')[0]]}, {inputs_label(k)})"
                                    for c in ['overall'] + REGIMES]
save(table3, 'table3_ann_by_regime',
     'ANN per-trajectory angular error (deg) on the held-out test set, overall and by flight regime, with the best baseline median in each regime for comparison.',
     'ann_regime', align='lrrrrrl')

,Regime,n trajectories,Median [95% CI],Mean,IQR,% > 15°,Best baseline (median)
0,Overall,1096,"5.0 [4.6, 5.4]",14.3,2.6–11.5,20.3,"8.3 (Möbius circular regression, air velocity)"
1,Upwind,721,"3.5 [3.2, 3.7]",7.9,2.2–6.5,9.8,"4.0 (Linear OLS, air velocity + ground velocity)"
2,Crosswind,639,"10.5 [9.6, 12.0]",20.0,5.4–21.6,39.1,"13.9 (Möbius circular regression, air velocity)"
3,Downwind,459,"9.8 [8.4, 12.1]",30.0,3.8–36.4,40.3,"23.8 (Circular OLS, thrust + air velocity + gr..."


## Table 4 — Leave-one-wind-speed-out

ANN: trained on two wind speeds and tested on the third (read from the saved results). The three baselines are refit here under the same leave-one-out design: the best pooled linear and circular OLS baselines and the airspeed Möbius regression. Baseline errors are medians over all frames of the held-out trajectories; ANN errors use the trajectories after the minimum-length filter, so counts can differ slightly.

In [6]:
lowo = pd.read_csv(OUT + 'leave_one_wind_out_results.csv')
# best linear and best circular OLS by pooled error, chosen as in 03_ann_vs_linear_models.ipynb
best_ols = {kind: base.loc[base.loc[base['model'].str.startswith(kind + ':'), 'overall'].idxmin(), 'model'].split(': ')[1]
            for kind in ('linear', 'circular')}
BASELINES = [('Linear OLS', tuple(best_ols['linear'].split('+')), 'linear'),
             ('Circular OLS', tuple(best_ols['circular'].split('+')), 'circular'),
             ('Möbius', ('air',), 'mobius')]
print('baselines:', [label for label, _, _ in BASELINES])
rows = []
for ws in sorted(data['windspeed'].unique()):
    trn, tst = data[data['windspeed'] != ws], data[data['windspeed'] == ws].reset_index(drop=True)
    res = {}
    for label, combo, kind in BASELINES:
        ang = mobius_angle(trn, tst, combo[0]) if kind == 'mobius' else ols_angle(trn, tst, combo, kind)
        s, _, _ = summarize(frame_errors(tst, ang)); res[label] = s
    for reg in ['overall'] + REGIMES:
        a = lowo[(np.isclose(lowo['held_out_windspeed'], ws)) & (lowo['regime'] == reg)].iloc[0]
        rows.append({'Held-out wind speed (m/s)': ws, 'Regime': reg.capitalize(), 'n (ANN)': int(a['n_trajectories']),
                     'ANN [95% CI]': ci_str(a['median'], a['ci95_lo'], a['ci95_hi']),
                     **{label: f'{res[label][reg]:.1f}' for label, _, _ in BASELINES}})
table4 = pd.DataFrame(rows)
save(table4, 'table4_leave_one_wind_out',
     'Leave-one-wind-speed-out median per-trajectory angular error (deg). The ANN and the three baselines are trained on two wind speeds and tested on the held-out third. '
     f"Baseline inputs: linear OLS, {inputs_label('linear: ' + best_ols['linear'])}; circular OLS, {inputs_label('circular: ' + best_ols['circular'])}; M\\\"obius circular regression, air velocity.",
     'lowo', align='lllrrrr')

baselines: ['Linear OLS', 'Circular OLS', 'Möbius']


,Held-out wind speed (m/s),Regime,n (ANN),ANN [95% CI],Linear OLS,Circular OLS,Möbius
0,0.3,Overall,1266,"6.0 [5.6, 6.4]",17.8,10.9,8.5
1,0.3,Upwind,750,"4.4 [4.1, 4.9]",5.4,4.9,6.9
2,0.3,Crosswind,771,"11.8 [11.0, 13.0]",25.7,26.2,12.3
3,0.3,Downwind,576,"11.8 [9.5, 14.6]",47.3,20.4,19.6
4,0.4,Overall,1060,"4.7 [4.4, 5.2]",9.6,10.2,8.4
5,0.4,Upwind,679,"3.1 [2.8, 3.4]",4.1,3.8,5.1
6,0.4,Crosswind,642,"10.0 [9.1, 11.0]",17.2,23.9,14.4
7,0.4,Downwind,422,"11.2 [9.3, 14.1]",37.4,26.6,24.5
8,0.6,Overall,995,"6.5 [5.9, 7.0]",7.6,9.6,6.9
9,0.6,Upwind,723,"3.8 [3.6, 4.0]",4.4,3.9,3.8


## Table 5 — Robustness across training seeds

In [7]:
seeds = pd.read_csv(OUT + 'ann_multiseed_results.csv')
t5 = pd.DataFrame({'Seed': seeds['seed'], 'Median error (deg)': seeds['median_error'].map('{:.2f}'.format),
                   'Mean error (deg)': seeds['mean_error'].map('{:.2f}'.format)})
summary = pd.DataFrame([{'Seed': 'mean ± sd', 'Median error (deg)': f"{seeds['median_error'].mean():.2f} ± {seeds['median_error'].std():.2f}",
                         'Mean error (deg)': f"{seeds['mean_error'].mean():.2f} ± {seeds['mean_error'].std():.2f}"},
                        {'Seed': 'range', 'Median error (deg)': f"{seeds['median_error'].min():.2f}–{seeds['median_error'].max():.2f}",
                         'Mean error (deg)': f"{seeds['mean_error'].min():.2f}–{seeds['mean_error'].max():.2f}"}])
table5 = pd.concat([t5, summary], ignore_index=True)
save(table5, 'table5_multiseed', 'ANN performance across five independently seeded training runs on the held-out test set.',
     'multiseed', align='lrr')

,Seed,Median error (deg),Mean error (deg)
0,42,4.96,14.27
1,7,5.04,14.14
2,13,5.04,14.03
3,99,5.02,14.27
4,123,4.92,14.22
5,mean ± sd,5.00 ± 0.05,14.19 ± 0.10
6,range,4.92–5.04,14.03–14.27


## Table 6 — Architecture selection (3-fold trajectory-grouped CV)

Out-of-fold median per-trajectory error with 95% bootstrap CI and IQR for each candidate architecture, trained with the geodesic loss and trajectory-length sample weights.

In [8]:
def cv_rows(df, label_fn):
    return [{'Configuration': label_fn(r), 'n': int(r['n_trajectories']),
             'CV median [95% CI]': ci_str(r['cv_median_error'], r['cv_ci95_lo'], r['cv_ci95_hi']),
             'IQR': f"{r['cv_q25']:.1f}–{r['cv_q75']:.1f}"} for _, r in df.iterrows()]

arch = lambda r: f"{int(r['layers'])} × {int(r['neurons'])}"
if os.path.exists(OUT + 'architecture_cv_results.csv'):
    table6 = pd.DataFrame(cv_rows(pd.read_csv(OUT + 'architecture_cv_results.csv'), arch))
    save(table6, 'table6_architecture_selection',
         'Grouped cross-validation results used to select the ANN architecture (geodesic loss, trajectory-length sample weights).',
         'selection', align='lrrr')
else:
    print('Skipping Table 6: architecture_cv_results.csv not found (run 02_training_and_testing_the_ann.ipynb with RETRAIN = True).')

,Configuration,n,CV median [95% CI],IQR
0,3 × 30,2225,"5.1 [4.8, 5.5]",2.7–12.5
1,3 × 20,2225,"5.3 [5.0, 5.6]",2.9–12.7
2,3 × 15,2225,"5.3 [5.0, 5.6]",2.8–12.8
3,2 × 20,2225,"5.3 [5.0, 5.6]",2.9–12.7
4,2 × 15,2225,"5.3 [5.1, 5.7]",2.9–12.7
5,3 × 10,2225,"5.4 [5.1, 5.6]",2.9–13.3
6,1 × 30,2225,"5.5 [5.3, 5.8]",3.0–13.9
7,1 × 20,2225,"5.5 [5.3, 5.8]",3.1–13.5
8,2 × 10,2225,"5.6 [5.3, 5.9]",2.9–13.6
9,2 × 30,2225,"5.6 [5.3, 5.8]",3.2–12.9


## Table 7 — Ensemble-agreement calibration

In [9]:
ens = pd.read_csv(OUT + 'ann_ensemble_agreement.csv')
ens['quartile'] = pd.qcut(ens['R'], 4, labels=['Q1 (lowest agreement)', 'Q2', 'Q3', 'Q4 (highest agreement)'])
table7 = ens.groupby('quartile', observed=True).agg(n=('err', 'size'), median_R=('R', 'median'),
                                                     median_err=('err', 'median'), mean_err=('err', 'mean')).reset_index()
rho = spearmanr(ens['R'], ens['err']).statistic
table7.columns = ['Agreement quartile', 'n', 'Median R', 'Median error (deg)', 'Mean error (deg)']
table7['Median R'] = table7['Median R'].map('{:.5f}'.format)
for c in ['Median error (deg)', 'Mean error (deg)']: table7[c] = table7[c].map('{:.1f}'.format)
print(f'Spearman rho (R vs error) = {rho:.3f}')
save(table7, 'table7_ensemble_calibration',
     f'Per-trajectory error by quartile of five-model ensemble mean resultant length $R$ (Spearman $\\rho={rho:.2f}$).',
     'calibration', align='lrrrr')

Spearman rho (R vs error) = -0.602


,Agreement quartile,n,Median R,Median error (deg),Mean error (deg)
0,Q1 (lowest agreement),274,0.99869,14.6,31.2
1,Q2,274,0.99966,6.1,12.3
2,Q3,274,0.99981,4.2,9.6
3,Q4 (highest agreement),274,0.99990,2.4,4.0


## Table 8 — ANN error vs. trajectory covariates

Spearman rank correlation between per-trajectory ANN error and trajectory-level kinematic summaries on the held-out set, and median error by wind speed.

In [10]:
cov = pd.read_csv(OUT + 'ann_error_with_covariates.csv')
labels = {'mean_slip_angle': 'Mean |slip angle| (deg)', 'max_turning_rate': 'Max turning rate (deg/frame)',
          'mean_thrust': 'Mean thrust magnitude', 'mean_groundspeed': 'Mean groundspeed (m/s)',
          'mean_airspeed': 'Mean airspeed (m/s)', 'windspeed': 'Wind speed (m/s)'}
rows = []
for c, lab in labels.items():
    r = spearmanr(cov[c], cov['err'])
    rows.append({'Covariate': lab, 'Spearman ρ': f'{r.statistic:.3f}', 'p': f'{r.pvalue:.1e}', 'n': len(cov)})
table8 = pd.DataFrame(rows).sort_values('Spearman ρ', key=lambda s: s.astype(float).abs(), ascending=False).reset_index(drop=True)
save(table8, 'table8_covariate_correlations',
     'Spearman correlation between per-trajectory ANN error and trajectory covariates on the held-out test set.',
     'covariates', align='lrrr')
ws_tab = cov.groupby('windspeed')['err'].agg(n='size', median='median', mean='mean').round(2).reset_index()
display(ws_tab)

,Covariate,Spearman ρ,p,n
0,Mean |slip angle| (deg),0.703,8.2e-164,1096
1,Max turning rate (deg/frame),0.587,2.7e-102,1096
2,Mean groundspeed (m/s),-0.545,1.1e-85,1096
3,Mean airspeed (m/s),-0.470,2.4e-61,1096
4,Mean thrust magnitude,-0.430,2.0e-50,1096
5,Wind speed (m/s),-0.004,8.9e-01,1096


,windspeed,n,median,mean
0,0.3,396,4.97,14.09
1,0.4,368,4.92,14.19
2,0.6,332,5.15,14.57


## Table 9 — Best baselines and the ANN on all held-out frames and on confident frames (Figure 6)

The values of `generate_figures/generate_figure_6.ipynb`: the best linear OLS, circular OLS and Möbius baselines (lowest overall error on all held-out frames, as in Table 1) and the ANN, scored on all held-out frames (Figure 6's wide bars) and only on the held-out frames with `heading_confidence > 0.99` (its narrow bars). Nothing is refit for the confident frames. Each trajectory's error is the median over its (confident) frames in the group; trajectories without such a frame drop out, so the confident rows rest on fewer trajectories (given in the caption). Median over trajectories with 95% bootstrap CI.

In [11]:
CONF_THRESHOLD = 0.99
assert len(ann_frames) == len(test_df) and (ann_frames['trajec_objid'].values == test_df['trajec_objid'].values).all(), \
    'tables/ann_by_regime.csv does not match this split: re-run 02_training_and_testing_the_ann.ipynb on the current data'
conf_mask = (test_df['heading_confidence'] > CONF_THRESHOLD).values

def predict(key):
    kind, inputs = key.split(': ')
    return mobius_angle(train_df, test_df, inputs) if kind == 'mobius' else ols_angle(train_df, test_df, tuple(inputs.split('+')), kind)

best9 = [nb_base.loc[nb_base.loc[nb_base['model'].str.startswith(fam + ':'), 'overall'].idxmin(), 'model']
         for fam in ('linear', 'circular', 'mobius')]
print('best baselines:', best9)
fig6_frames = {**{k: frame_errors(test_df, predict(k)) for k in best9}, 'ANN': ann_frames}

def group_errors(fdf):
    _, pooled, by_reg = summarize(fdf)
    return {'overall': pooled.values, **{r: by_reg.loc[by_reg['regime'] == r, 'err'].values for r in REGIMES}}

rows, n_traj = [], {}
for key, fdf in fig6_frames.items():
    for which, f in (('All', fdf), (f'Confidence > {CONF_THRESHOLD:g}', fdf[conf_mask])):
        g = group_errors(f)
        rows.append({'Model': model_label(key), 'Frames': which, **{r.capitalize(): ci_str(*bootstrap_ci(v)) for r, v in g.items()}})
        n_traj[which] = {r: len(v) for r, v in g.items()}   # the same for every model
table9 = pd.DataFrame(rows)
n_str = ', '.join(f"{r} {n_traj['All'][r]} ({n_traj[f'Confidence > {CONF_THRESHOLD:g}'][r]})" for r in ['overall'] + REGIMES)
save(table9, 'table9_confident_frames',
     'Median per-trajectory angular error (deg) [95\\% CI] of the best linear, circular and M\\"obius baselines and of the ANN on all held-out frames '
     f'and on the held-out frames with heading confidence $> {CONF_THRESHOLD:g}$ (Figure 6). Nothing is refit for the confident frames. '
     f'Held-out trajectories, all (confident): {n_str}.',
     'confident_frames', align='llrrrr')

best baselines: ['linear: thrust+air+ground', 'circular: air+ground', 'mobius: air']


,Model,Frames,Overall,Upwind,Crosswind,Downwind
0,Linear OLS (thrust + air velocity + ground vel...,All,"9.5 [8.6, 10.5]","4.3 [4.1, 4.6]","17.5 [16.2, 19.1]","35.2 [32.4, 38.3]"
1,Linear OLS (thrust + air velocity + ground vel...,Confidence > 0.99,"7.0 [6.4, 7.7]","4.2 [3.9, 4.4]","15.0 [13.8, 16.4]","22.7 [19.8, 25.9]"
2,Circular OLS (air velocity + ground velocity),All,"9.4 [8.4, 10.7]","4.1 [3.7, 4.4]","22.6 [21.0, 23.9]","25.3 [21.8, 29.1]"
3,Circular OLS (air velocity + ground velocity),Confidence > 0.99,"7.1 [6.6, 8.1]","3.9 [3.6, 4.2]","20.8 [19.4, 22.0]","19.1 [15.7, 21.9]"
4,Möbius circular regression (air velocity),All,"8.3 [7.8, 9.0]","5.4 [5.0, 5.8]","13.9 [12.6, 14.9]","24.3 [21.5, 27.9]"
5,Möbius circular regression (air velocity),Confidence > 0.99,"7.0 [6.4, 7.3]","5.2 [4.8, 5.6]","12.2 [10.9, 13.2]","16.3 [14.7, 19.8]"
6,ANN,All,"5.0 [4.6, 5.4]","3.5 [3.2, 3.7]","10.5 [9.6, 12.0]","9.8 [8.4, 12.1]"
7,ANN,Confidence > 0.99,"4.5 [4.1, 4.8]","3.3 [3.1, 3.6]","8.9 [7.9, 10.1]","7.5 [6.1, 9.1]"
